#### 29.10.25, &copy; [Evhenii Kostin](https://github.com/DE123MasterProgram2025autumn/DE_Kostin), 2025

# Лабораторна робота №5: Функціональне програмування та паралельні обчислення в R


__Мета:__ _освоїти сучасні підходи до функціонального програмування за допомогою пакета purrr та навчитися масштабувати обчислення через багатопоточність (foreach, furrr, future). Навчитися будувати ефективні, читабельні та масштабовані конвеєри обробки даних, які можуть працювати з великими обсягами інформації або довготривалими операціями._

### Варіант 2
#### Тема: Масштабований аналіз даних про веб-трафік

In [6]:
# --- 0. ПІДКЛЮЧЕННЯ БІБЛІОТЕК ТА НАЛАШТУВАННЯ ---
# Встановлюємо всі необхідні пакети
if (!require("tidyverse")) install.packages("tidyverse")
if (!require("jsonlite")) install.packages("jsonlite")
if (!require("furrr")) install.packages("furrr")
if (!require("future")) install.packages("future")
if (!require("foreach")) install.packages("foreach")
if (!require("doFuture")) install.packages("doFuture")

library(tidyverse) # Містить purrr, dplyr, readr та ін.
library(jsonlite)  # Для роботи з JSON
library(furrr)     # Паралельна версія purrr
library(future)    # Основа для паралельних обчислень
library(foreach)   # Для циклів foreach
library(doFuture)  # Бекенд для foreach

# --- ГЛОБАЛЬНЕ НАЛАШТУВАННЯ ПАРАЛЕЛЬНОСТІ ---
# Встановлюємо план 'multisession' - R запустить фонові процеси
plan(multisession)
# Реєструємо doFuture як бекенд для %dopar%
registerDoFuture()


# --- ЕТАП 0: СИМУЛЯЦІЯ ДАНИХ ---
message("--- ЕТАП 0: СТВОРЕННЯ ВИХІДНИХ ФАЙЛІВ ---")
data_dir <- "data/traffic"
dir.create(data_dir, recursive = TRUE, showWarnings = FALSE)

# Базовий шаблон для "хорошого" JSON-файлу
traffic_template <- list(
  list(
    date = "2024-01-05", page_views = 1250, unique_users = 840, bounce_rate = 0.42,
    campaigns = list(
      list(name = "google_ads", channel = "paid_search", cost = 120),
      list(name = "email_jan", channel = "email", cost = 50)
    )
  ),
  list(
    date = "2024-01-15", page_views = 1300, unique_users = 900, bounce_rate = 0.40,
    campaigns = list(
      list(name = "google_ads", channel = "paid_search", cost = 130),
      list(name = "facebook_jan", channel = "social", cost = 0) # Вартість 0
    )
  )
)

# Створимо 10 файлів
file_paths <- file.path(data_dir, paste0("traffic_", sprintf("%02d", 1:10), ".json"))

# Створимо 8 хороших файлів
walk(file_paths[1:8], ~ {
  jsonlite::write_json(traffic_template, .x, auto_unbox = TRUE, pretty = TRUE)
})

# 9. Створимо НЕПОВНИЙ ФАЙЛ (відсутній 'campaigns')
broken_data_1 <- list(
  list(
    date = "2024-02-05", page_views = 1100, unique_users = 750, bounce_rate = 0.45
  )
)
jsonlite::write_json(broken_data_1, file_paths[9], auto_unbox = TRUE, pretty = TRUE)

# 10. Створимо ПОШКОДЖЕНИЙ ФАЙЛ (невалідний JSON)
writeLines("{'date': '2024-03-01', ... ", file_paths[10])

message(paste("Створено 10 файлів у", data_dir))
message("--- ЕТАП 0 ЗАВЕРШЕНО. ---\n")

--- ЕТАП 0: СТВОРЕННЯ ВИХІДНИХ ФАЙЛІВ ---

Створено 10 файлів у data/traffic

--- ЕТАП 0 ЗАВЕРШЕНО. ---




### Етап 1: Функціональне завантаження та розгортання (purrr + jsonlite)
- Створити функцію safe_read_traffic(path), яка використовує tryCatch() для безпечного читання JSON.
- Використати hoist() та unnest_longer() для виділення:
- - date, page_views, unique_users, bounce_rate
- - campaign_name, channel, cost з вкладеного масиву campaigns
- Використати map() для завантаження всіх файлів.
- Відфільтрувати записи, де вартість кампанії > 0, за допомогою keep().

In [8]:
# --- Етап 1: Функціональне завантаження та розгортання (purrr + jsonlite) ---
message("--- ЕТАП 1: ФУНКЦІОНАЛЬНЕ ЗАВАНТАЖЕННЯ ---")

#' Безпечно читає та розгортає JSON-файл з даними про трафік
#' Використовує `unnest` для розгортання `campaigns`
#'
#' @param path Шлях до JSON-файлу.
#' @return tibble з 7 стовпцями або NULL у разі помилки.
safe_read_traffic <- function(path) {
  tryCatch({
    # simplifyVector = TRUE автоматично перетворює JSON на data.frame
    data <- jsonlite::read_json(path, simplifyVector = TRUE) %>%
      as_tibble()
    
    # Перевірка на неповні дані (напр., файл №9)
    if (!"campaigns" %in% names(data)) {
      message(paste("⚠️ Пропущено (відсутній 'campaigns'):", basename(path)))
      return(NULL)
    }
    
    # Розгортаємо вкладені кампанії
    data %>%
      tidyr::unnest(campaigns) %>%
      # Вибираємо та перейменовуємо поля згідно завдання
      select(
        date, page_views, unique_users, bounce_rate,
        campaign_name = name, channel, cost
      )
      
  }, error = function(e) {
    # Обробка пошкодженого JSON (файл №10)
    message(paste("❗ Помилка читання JSON:", basename(path), "-", e$message))
    return(NULL)
  })
}

# Отримуємо список усіх JSON-файлів
all_file_paths <- list.files(data_dir, pattern = "\\.json$", full.names = TRUE)

# 1. map() для завантаження всіх файлів
list_of_data <- map(all_file_paths, safe_read_traffic)

# Додаємо імена файлів до списку
names(list_of_data) <- basename(all_file_paths)

# 2. Фільтруємо NULL (пошкоджені/неповні файли)
list_of_data_compact <- purrr::compact(list_of_data)

# 3. Фільтруємо ЗАПИСИ (rows) де cost > 0
list_of_data_filtered <- map(list_of_data_compact, ~ .x %>% filter(cost > 0))

message(paste("Успішно завантажено та відфільтровано:", length(list_of_data_filtered), "файлів."))

--- ЕТАП 1: ФУНКЦІОНАЛЬНЕ ЗАВАНТАЖЕННЯ ---

⚠️ Пропущено (відсутній 'campaigns'): traffic_09.json

❗ Помилка читання JSON: traffic_10.json - lexical error: invalid char in json text.
                                      {'date': '2024-03-01', ...  
                     (right here) ------^


Успішно завантажено та відфільтровано: 8 файлів.



### Етап 2: Послідовна агрегація (reduce)
- Об’єднати всі таблиці в єдину за допомогою reduce(bind_rows).
- Додати стовпець file_id на основі імені файлу.

In [3]:
# --- Етап 2: Послідовна агрегація (reduce) ---
message("\n--- ЕТАП 2: ПОСЛІДОВНА АГРЕГАЦІЯ ---")

# Щоб додати `file_id` (згідно завдання) ПЕРЕД `reduce`,
# ми використовуємо `imap` (index-map) для додавання імені файлу у стовпець.
list_with_ids <- imap(list_of_data_filtered, ~ .x %>% mutate(file_id = .y))

# Тепер об'єднуємо всі таблиці в одну за допомогою reduce
all_traffic_data <- reduce(list_with_ids, bind_rows)

message("Результат Етапу 2 (all_traffic_data):")
print(head(all_traffic_data))


--- ЕТАП 2: ПОСЛІДОВНА АГРЕГАЦІЯ ---

Результат Етапу 2 (all_traffic_data):

# A tibble: 6 × 8
  date   page_views unique_users bounce_rate campaign_name channel  cost file_id
  <chr>       <int>        <int>       <dbl> <chr>         <chr>   <int> <chr>  
1 2024-…       1250          840        0.42 google_ads    paid_s…   120 traffi…
2 2024-…       1250          840        0.42 email_jan     email      50 traffi…
3 2024-…       1300          900        0.4  google_ads    paid_s…   130 traffi…
4 2024-…       1250          840        0.42 google_ads    paid_s…   120 traffi…
5 2024-…       1250          840        0.42 email_jan     email      50 traffi…
6 2024-…       1300          900        0.4  google_ads    paid_s…   130 traffi…


### Етап 3: Паралельний аналіз (foreach + furrr)
- Написати функцію analyze_campaign(df), яка для кожної кампанії рахує:
- - загальну вартість
- - середній bounce rate
- - ROI = (unique_users * 0.1 - cost) / cost
- Виконати цей аналіз двома способами:
1. За допомогою foreach(..., .combine = bind_rows) %dopar%.
2. За допомогою future_map_dfr() з furrr.

In [4]:
# --- Етап 3: Паралельний аналіз (foreach + furrr) ---
message("\n--- ЕТАП 3: ПАРАЛЕЛЬНИЙ АНАЛІЗ ---")

#' Функція аналізу для одного фрагмента даних (одна кампанія)
#'
#' @param df_chunk tibble з даними *лише для однієї* кампанії
#' @return tibble з 1 рядка з результатами
analyze_campaign_chunk <- function(df_chunk) {
  # Симулюємо "важку" роботу, щоб паралелізм був помітний
  Sys.sleep(0.1) 
  
  data_summary <- df_chunk %>%
    summarise(
      total_cost = sum(cost, na.rm = TRUE),
      avg_bounce_rate = mean(bounce_rate, na.rm = TRUE),
      total_revenue = sum(unique_users * 0.1, na.rm = TRUE) # Умовний дохід
    )
  
  # Розрахунок ROI
  data_summary %>%
    mutate(
      ROI = (total_revenue - total_cost) / total_cost,
      # Додаємо назву кампанії назад
      campaign_name = df_chunk$campaign_name[1]
    ) %>%
    # Перевпорядковуємо стовпці для краси
    select(campaign_name, ROI, total_cost, avg_bounce_rate)
}

# Створюємо список, де кожен елемент - це df для однієї кампанії
list_of_campaign_dfs <- all_traffic_data %>%
  group_split(campaign_name) # Розділяє df на список по групах

message(paste("Починаємо паралельний аналіз для", length(list_of_campaign_dfs), "кампаній..."))

# --- 3a. Аналіз за допомогою foreach ---
message("3a. Виконуємо `foreach`...")
start_time_foreach <- Sys.time()

analysis_foreach <- foreach(
  campaign_df = list_of_campaign_dfs,
  .combine = bind_rows # Як збирати результати
) %dopar% {
  # Цей код виконується паралельно
  analyze_campaign_chunk(campaign_df)
}

end_time_foreach <- Sys.time()
message(paste("Foreach зайняв:", round(end_time_foreach - start_time_foreach, 2), "секунд"))
print(analysis_foreach)

# --- 3b. Аналіз за допомогою furrr ---
message("\n3b. Виконуємо `future_map_dfr` (furrr)...")
start_time_furrr <- Sys.time()

# future_map_dfr - це паралельний map, який одразу збирає в data.frame
analysis_furrr <- future_map_dfr(
  list_of_campaign_dfs,
  analyze_campaign_chunk,
  .options = furrr_options(seed = TRUE) # Для відтворюваності
)

end_time_furrr <- Sys.time()
message(paste("Furrr зайняв:", round(end_time_furrr - start_time_furrr, 2), "секунд"))
print(analysis_furrr)


--- ЕТАП 3: ПАРАЛЕЛЬНИЙ АНАЛІЗ ---

Починаємо паралельний аналіз для 2 кампаній...

3a. Виконуємо `foreach`...

Foreach зайняв: 0.84 секунд

# A tibble: 2 × 4
  campaign_name    ROI total_cost avg_bounce_rate
  <chr>          <dbl>      <int>           <dbl>
1 email_jan      0.68         400            0.42
2 google_ads    -0.304       2000            0.41

3b. Виконуємо `future_map_dfr` (furrr)...

Furrr зайняв: 0.26 секунд

# A tibble: 2 × 4
  campaign_name    ROI total_cost avg_bounce_rate
  <chr>          <dbl>      <int>           <dbl>
1 email_jan      0.68         400            0.42
2 google_ads    -0.304       2000            0.41


### Етап 4: Просунута багатопоточність (future)
- Створити вкладені ф’ючерси:
- - Зовнішній — для кожного файлу.
- - Внутрішній — для обробки кожної кампанії (наприклад, валідація ROI).
- Використати plan(list(multisession, multisession)).
- Додати обробку помилок через resolved() та backtrace().

In [16]:
# --- Етап 4: Просунута багатопоточність (future) ---
message("\n--- ЕТАП 4: ПРОСУНУТА БАГАТОПОТОЧНІСТЬ (ВКЛАДЕНІ FUTURE) ---")

# ----------------------------------------------------------------------
# 💡 ВИПРАВЛЕННЯ №1:
# Прибираємо `tweak(workers = 4)`, щоб R автоматично
# використав доступні ядра (2 у вашому випадку) і не видавав попередження.
# ----------------------------------------------------------------------
plan(list(
  multisession, # Зовнішній план (автоматична к-ть ядер)
  sequential    # Внутрішній план (послідовно)
))

message(paste("Встановлено вкладений план: multisession / sequential"))


# Внутрішня функція (симулює валідацію 1 кампанії)
validate_roi_inner <- function(campaign_row) {
    Sys.sleep(0.1) # Симуляція роботи
    roi <- (campaign_row$unique_users * 0.1 - campaign_row$cost) / campaign_row$cost
    
    # Симуляція помилки для демонстрації
    if (campaign_row$campaign_name == "email_jan" && runif(1) > 0.5) {
      stop("Помилка валідації ROI для email_jan!")
    }
    
    list(campaign = campaign_row$campaign_name, roi = roi, status = "OK")
}

# Зовнішня функція (обробка 1 файлу)
process_file_outer <- function(path) {
  # Це "зовнішній" future (запускається паралельно)
  future({
    message(paste("-> Починаємо обробку файлу:", basename(path)))
    # Використовуємо функцію з Етапу 1
    data <- safe_read_traffic(path)
    if (is.null(data) || nrow(data) == 0) return(list())
    
    # Розбиваємо df на список рядків для "внутрішніх" завдань
    rows_as_list <- split(data, seq(nrow(data)))
    
    # Використовуємо звичайний `map()` (він виконується послідовно всередині future)
    # і одразу ловимо помилки
    inner_results <- map(rows_as_list, ~{
      tryCatch(
        validate_roi_inner(.x),
        error = function(e) { e } # Ловимо помилку "email_jan"
      )
    })
    
    return(inner_results)
    
  }, seed = TRUE)
}

# Запускаємо вкладений процес
message("Запускаємо вкладені ф'ючерси...")
nested_futures <- map(all_file_paths[1:4], process_file_outer) # Обмежимо 4 файлами

# Збираємо результати. `future::value()` безпечно збирає результати
nested_results <- future::value(nested_futures)

message("\nОбробка помилок у вкладених ф'ючерсах:")
all_results <- unlist(nested_results, recursive = FALSE)

# ----------------------------------------------------------------------
# 💡 ВИПРАВЛЕННЯ №2:
# Додаємо `.else = ~NULL` до обох `map_if`.
# Це гарантує, що стовпці `data` та `error_msg`
# містять лише сумісні типи (списки/NULL або символи/NULL).
# ----------------------------------------------------------------------
results_df <- tibble(
  raw = all_results
) %>%
  mutate(
    # Перевіряємо, чи результат - це помилка (або "condition")
    is_error = map_lgl(raw, ~inherits(.x, "condition")),
    # Витягуємо або результат, або NULL
    data = map_if(raw, !is_error, ~.x, .else = ~NULL), # <--- ЗМІНА (додано .else)
    # Витягуємо повідомлення про помилку
    error_msg = map_if(raw, is_error, ~as.character(.x$message), .else = ~NULL) # <--- ЗМІНА (додано .else)
  )

# Тепер вивід покаже і успішні, і провалені завдання
print(results_df %>% select(-raw) %>% unnest_wider(data) %>% unnest(error_msg, keep_empty = TRUE))

# Повертаємо план до звичайного
plan(multisession)


--- ЕТАП 4: ПРОСУНУТА БАГАТОПОТОЧНІСТЬ (ВКЛАДЕНІ FUTURE) ---

Встановлено вкладений план: multisession / sequential

Запускаємо вкладені ф'ючерси...

-> Починаємо обробку файлу: traffic_01.json

-> Починаємо обробку файлу: traffic_02.json

-> Починаємо обробку файлу: traffic_03.json

-> Починаємо обробку файлу: traffic_04.json


Обробка помилок у вкладених ф'ючерсах:

# A tibble: 16 × 5
   is_error campaign         roi status error_msg                           
   <lgl>    <chr>          <dbl> <chr>  <chr>                               
 1 FALSE    google_ads    -0.3   OK     NA                                  
 2 TRUE     NA            NA     NA     Помилка валідації ROI для email_jan!
 3 FALSE    google_ads    -0.308 OK     NA                                  
 4 FALSE    facebook_jan Inf     OK     NA                                  
 5 FALSE    google_ads    -0.3   OK     NA                                  
 6 TRUE     NA            NA     NA     Помилка валідації ROI для emai

### Етап 5: Збереження результатів
- Зберегти високоефективні кампанії (ROI > 0.5) у файли за допомогою walk2() (наприклад, high_roi_01.csv, high_roi_02.csv, …).

In [17]:
# --- Етап 5: Збереження результатів (walk) ---
message("\n--- ЕТАП 5: ЗБЕРЕЖЕННЯ РЕЗУЛЬТАТІВ ---")

# 1. Визначимо високоприбуткові кампанії (використовуємо `analysis_furrr` з Етапу 3)
high_roi_campaign_names <- analysis_furrr %>%
  filter(ROI > 0.5) %>%
  pull(campaign_name)

if (length(high_roi_campaign_names) > 0) {
  
  # 2. Фільтруємо вихідні дані (`all_traffic_data` з Етапу 2) за цими кампаніями
  data_to_save <- all_traffic_data %>%
    filter(campaign_name %in% high_roi_campaign_names) %>%
    # Розділяємо на список (один df на кампанію)
    group_split(campaign_name) %>%
    # Називаємо елементи списку для iwalk
    set_names(high_roi_campaign_names)

  # 3. Створюємо папку
  reports_dir <- "data/high_roi_reports"
  dir.create(reports_dir, showWarnings = FALSE)

  # 4. Використовуємо iwalk() (indexed-walk) для збереження
  # .x - це дані (df), .y - це назва (ім'я зі списку)
  iwalk(data_to_save, ~{
    file_name <- paste0("high_roi_", .y, ".csv")
    file_path <- file.path(reports_dir, file_name)
    
    message(paste("...Збереження:", file_name))
    write_csv(.x, file_path)
  })
  
  message(paste("Збережено", length(data_to_save), "звіт(ів) у папку", reports_dir))
  
} else {
  message("Високоефективних кампаній (ROI > 0.5) не знайдено.")
}

message("\n--- ЛАБОРАТОРНУ РОБОТУ №5 ЗАВЕРШЕНО ---")

# --- Очищення (необов'язково) ---
# unlink("data", recursive = TRUE)


--- ЕТАП 5: ЗБЕРЕЖЕННЯ РЕЗУЛЬТАТІВ ---

...Збереження: high_roi_email_jan.csv

Збережено 1 звіт(ів) у папку data/high_roi_reports


--- ЛАБОРАТОРНУ РОБОТУ №5 ЗАВЕРШЕНО ---



<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=6a083947-94ba-475d-8730-27cff0574f54' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>